### CRISP-DM Phase 4.3 - Modeling : Visualization dashboard

In [ ]:
import pandas as pd
from iso3166 import countries_by_alpha3

In [ ]:
# Load the datasets
legislative_coverage_country = pd.read_csv('data/country_legislative_coverage.csv')
hazard_intensity_country = pd.read_csv('data/country_hazard_intensity.csv')
correlation_country = pd.read_csv('data/country_correlation.csv')

legislative_coverage_continent = pd.read_csv('data/continent_legislative_coverage.csv')
hazard_intensity_continent = pd.read_csv('data/continent_hazard_intensity.csv')
correlation_continent = pd.read_csv('data/continent_correlation.csv')

In [ ]:
VARIABLES = ['2m_temperature', 'Instantaneous_wind_gust', 'Sea_level_anomaly', 
             'Snowmelt', 'SPEI', 'Total_precipitation']

hazard_variable_dict = {'flood': 'Total_precipitation', 'drought': 'SPEI', 
                        'temperature_extremes': '2m_temperature', 'sea_level_rise': 'Sea_level_anomaly', 
                        'storm': 'Instantaneous_wind_gust', 'melting': 'Snowmelt'}

hazards = list(hazard_variable_dict.keys())

Country dataframe

In [ ]:
def iso_to_name(iso):
    try:
        result = countries_by_alpha3.get(iso)
        return result.name if result else None
    except:
        return None

country_fix_dict = {
    'Bolivia, Plurinational State of': 'Bolivia',
    'Venezuela, Bolivarian Republic of': 'Venezuela',
    'Iran, Islamic Republic of': 'Iran',
    "Côte d'Ivoire": 'Ivory Coast',
    'Congo, Democratic Republic of the': 'Democratic Republic of the Congo',
    'Congo': 'Republic of the Congo',
    'Tanzania, United Republic of': 'Tanzania',
    'Türkiye': 'Turkey',
    'Moldova, Republic of': 'Moldova',
    'Taiwan, Province of China': 'Taiwan',
    'Palestine, State of': 'Palestine'
}

In [ ]:
# Merge legislative coverage and hazard intensity
country = legislative_coverage_country.merge(hazard_intensity_country[['Country', 'Year'] + [v for v in VARIABLES]], on=['Country', 'Year'], how='inner')
country.rename(columns={'Count': 'Coverage'}, inplace=True)

# Keep intensity values only for the relevant hazard variable
country['Intensity'] = country.apply(
    lambda row: row[hazard_variable_dict.get(row['Hazard'], '')] 
    if hazard_variable_dict.get(row['Hazard']) else None, axis=1)
country.drop(columns=VARIABLES, inplace=True)

# Merge with correlation data
country = country.merge(correlation_country, on=['Country', 'Hazard'], how='left')

# Add country names
country['Country_name'] = country['Country'].apply(iso_to_name)
country['Country_name'] = country['Country_name'].replace(country_fix_dict)

In [ ]:
## Export .csv
dashboard_data = country[['Country', 'Country_name', 'Year', 'Hazard', 'Coverage', 'Intensity', 'Rho', 'p_value']]
dashboard_data = dashboard_data[dashboard_data['Hazard'].isin(hazards)].copy()
dashboard_data.to_csv('outputs/dashboard_data.csv', index=False)